In [1]:
import sys
sys.path.append("../src")

from data_loader import load_all
from lstm_utils import build_sequences
from fault_reference import HARD_TO_DETECT_FAULTS

train, test = load_all()

measurement_cols = [c for c in train.columns if c.startswith("XMEAS") or c.startswith("XMV")]

train_filtered = train[~train["faultNumber"].isin(HARD_TO_DETECT_FAULTS)]

X_train_seq, y_train_seq = build_sequences(train_filtered, measurement_cols, window=20)

print("X shape:", X_train_seq.shape)
print("y shape:", y_train_seq.shape)

X shape: (8779, 20, 52)
y shape: (8779,)


In [2]:
#test data
test_filtered = test[~test["faultNumber"].isin(HARD_TO_DETECT_FAULTS)]

X_test_seq, y_test_seq = build_sequences(test_filtered, measurement_cols, window=20)

print("X_test shape:", X_test_seq.shape)
print("y_test shape:", y_test_seq.shape)

X_test shape: (17879, 20, 52)
y_test shape: (17879,)


In [3]:
# standardizing and encodeing the fault labels. Standardization is necessary for lstm models to perform well. 
# The fault labels are encoded to integers for the model to predict.    

from sklearn.preprocessing import StandardScaler, LabelEncoder

n_samples, n_timesteps, n_features = X_train_seq.shape

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_seq.reshape(-1, n_features))
X_train_scaled = X_train_scaled.reshape(n_samples, n_timesteps, n_features)

n_test_samples = X_test_seq.shape[0]
X_test_scaled = scaler.transform(X_test_seq.reshape(-1, n_features))
X_test_scaled = X_test_scaled.reshape(n_test_samples, n_timesteps, n_features)

label_encoder_lstm = LabelEncoder()
y_train_encoded = label_encoder_lstm.fit_transform(y_train_seq)
y_test_encoded = label_encoder_lstm.transform(y_test_seq)

print("X_train_scaled shape:", X_train_scaled.shape)
print("Number of classes:", len(label_encoder_lstm.classes_))

X_train_scaled shape: (8779, 20, 52)
Number of classes: 19
